# PiCar Sensor Data Modeling

This notebook ingests logged sensor profiles from the PiCar and builds models to improve autonomous navigation.

## Models
1. **Speed Calibration** — motor % → actual cm/s (currently only 2 data points)
2. **Braking / Deceleration** — real deceleration curves (currently a guess: 200 cm/s²)
3. **Steering Response** — servo angle × speed → yaw rate (bicycle model)
4. **Terrain Compensation** — incline vs speed loss

## Workflow
1. Connect to Pico & download log (or load from file)
2. Parse into DataFrame
3. Run each model
4. Export updated `vehicle_profiles.json`

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from pathlib import Path

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['figure.dpi'] = 100
plt.style.use('seaborn-v0_8-whitegrid')

---
## 1. Load Sensor Log

Either download live from the Pico or load a previously saved JSON file.

In [ ]:
# Option A: Download from Pico (uncomment to use)
# from picar_client import PicarClient
# client = PicarClient()
# log_data = client.log_download(save_path='drive_session.json')

# Option B: Load from file
LOG_FILE = 'drive_session.json'  # <-- change to your file

with open(LOG_FILE) as f:
    log_data = json.load(f)

print(f"Profile:      {log_data['profile']}")
print(f"Version:      {log_data['version']}")
print(f"Interval:     {log_data['interval_ms']} ms")
print(f"Duration:     {log_data['duration_s']} s")
print(f"Samples:      {log_data['sample_count']}")
print(f"Fields:       {log_data['fields']}")

In [ ]:
# Parse into DataFrame
df = pd.DataFrame(log_data['samples'], columns=log_data['fields'])

# Convert timestamp to seconds
df['t'] = df['ts'] / 1000.0

# Compute dt between consecutive samples
df['dt'] = df['t'].diff().fillna(log_data['interval_ms'] / 1000.0)

print(f"DataFrame shape: {df.shape}")
df.head(10)

---
## 2. Data Overview & Quality

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(14, 10), sharex=True)

# Motor & Servo
axes[0, 0].plot(df['t'], df['motor'], 'b-', linewidth=0.8)
axes[0, 0].set_ylabel('Motor %')
axes[0, 0].set_title('Motor Speed')
axes[0, 0].axhline(y=0, color='k', linewidth=0.5)

axes[0, 1].plot(df['t'], df['servo'], 'r-', linewidth=0.8)
axes[0, 1].set_ylabel('Servo (°)')
axes[0, 1].set_title('Steering Angle')
axes[0, 1].axhline(y=90, color='k', linewidth=0.5, linestyle='--', label='centre')
axes[0, 1].legend()

# ToF distances
axes[1, 0].plot(df['t'], df['tof_l'], 'g-', linewidth=0.8, label='Left')
axes[1, 0].plot(df['t'], df['tof_r'], 'm-', linewidth=0.8, label='Right')
axes[1, 0].set_ylabel('Distance (cm)')
axes[1, 0].set_title('Front ToF Sensors')
axes[1, 0].legend()

# Ultrasonic rear
axes[1, 1].plot(df['t'], df['us_rear'], 'c-', linewidth=0.8)
axes[1, 1].set_ylabel('Distance (cm)')
axes[1, 1].set_title('Rear Ultrasonic')

# IMU: pitch & roll
axes[2, 0].plot(df['t'], df['pitch'], 'orange', linewidth=0.8, label='Pitch')
axes[2, 0].plot(df['t'], df['roll'], 'purple', linewidth=0.8, label='Roll')
axes[2, 0].set_ylabel('Angle (°)')
axes[2, 0].set_title('IMU Tilt')
axes[2, 0].set_xlabel('Time (s)')
axes[2, 0].legend()

# Gyroscope Z (yaw rate)
axes[2, 1].plot(df['t'], df['gz'], 'brown', linewidth=0.8)
axes[2, 1].set_ylabel('Yaw Rate (°/s)')
axes[2, 1].set_title('Gyroscope Z (Yaw)')
axes[2, 1].set_xlabel('Time (s)')

plt.tight_layout()
plt.suptitle('Drive Session Overview', y=1.02, fontsize=14, fontweight='bold')
plt.show()

# Data quality report
null_counts = df[['ax','ay','az','tof_l','tof_r','us_rear']].isnull().sum()
print('\nNull counts per sensor:')
print(null_counts)
print(f'\nSample rate jitter: mean={df["dt"].mean()*1000:.1f}ms, std={df["dt"].std()*1000:.1f}ms')

---
## 3. Model 1: Speed Calibration

**Goal:** Map motor % → actual speed (cm/s).

**Method:** Compute velocity from consecutive ToF distance readings, group by motor %, fit the sqrt curve from `motor.py`:

$$v = k \cdot \sqrt{\frac{\text{motor\%} - c}{100 - c}}$$

where $k$ = max speed, $c$ = PWM cutoff (5).

In [ ]:
# Use the minimum front ToF distance to estimate forward distance changes
# Only use samples where car is moving forward and steering roughly straight
speed_df = df.copy()
speed_df = speed_df[speed_df['motor'] > 5]  # forward only, above dead zone
speed_df = speed_df[speed_df['servo'].between(80, 100)]  # roughly straight
speed_df = speed_df.dropna(subset=['tof_l', 'tof_r'])

# Front distance = min(tof_l, tof_r) — distance to wall ahead
speed_df['front_dist'] = speed_df[['tof_l', 'tof_r']].min(axis=1)

# Velocity = negative of distance change rate (approaching wall = positive speed)
speed_df['velocity'] = -speed_df['front_dist'].diff() / speed_df['dt']

# Filter out noise: remove huge jumps (sensor glitches)
speed_df = speed_df[speed_df['velocity'].between(-5, 80)]  # reasonable range
speed_df = speed_df[speed_df['velocity'] > 0]  # only approaching

print(f"Usable samples for speed calibration: {len(speed_df)}")

if len(speed_df) > 10:
    # Group by motor % and compute mean velocity
    speed_by_motor = speed_df.groupby('motor')['velocity'].agg(['mean', 'std', 'count'])
    speed_by_motor = speed_by_motor[speed_by_motor['count'] >= 3]  # need at least 3 samples
    
    print('\nMeasured speed by motor %:')
    print(speed_by_motor.round(2))
else:
    print('⚠️  Not enough straight-line forward samples.')
    print('   Tip: Drive straight toward a wall at different motor speeds.')

In [ ]:
# Fit the sqrt speed model: v = k * sqrt((motor_pct - c) / (100 - c))
PWM_CUTOFF = 5  # from motor.py / motor2.py

def speed_model(motor_pct, k):
    """Sqrt speed mapping matching motor.py PWM formula."""
    norm = np.sqrt(np.maximum(0, (motor_pct - PWM_CUTOFF) / (100 - PWM_CUTOFF)))
    return k * norm

if len(speed_df) > 10 and len(speed_by_motor) >= 2:
    x_data = speed_by_motor.index.values.astype(float)
    y_data = speed_by_motor['mean'].values
    
    popt, pcov = curve_fit(speed_model, x_data, y_data, p0=[15.0])
    k_fit = popt[0]
    
    print(f'Fitted speed constant k = {k_fit:.2f} cm/s (at 100%)')
    print(f'  → speed at 100% = {speed_model(100, k_fit):.1f} cm/s')
    print(f'  → speed at  80% = {speed_model(80, k_fit):.1f} cm/s')
    print(f'  → speed at  50% = {speed_model(50, k_fit):.1f} cm/s')
    print(f'  → speed at  20% = {speed_model(20, k_fit):.1f} cm/s')
    
    # Plot
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.scatter(x_data, y_data, s=speed_by_motor['count'].values * 3, 
               c='steelblue', alpha=0.7, edgecolors='navy', label='Measured')
    
    x_smooth = np.linspace(PWM_CUTOFF, 100, 100)
    ax.plot(x_smooth, speed_model(x_smooth, k_fit), 'r-', linewidth=2, 
            label=f'Fit: v = {k_fit:.1f} · √((m-{PWM_CUTOFF})/95)')
    
    # Show current profile values for comparison
    ax.axhline(y=11.1, color='gray', linestyle='--', alpha=0.5, label='Current profile (100%=11.1)')
    
    ax.set_xlabel('Motor %')
    ax.set_ylabel('Speed (cm/s)')
    ax.set_title('Speed Calibration: Motor % → Actual Speed')
    ax.legend()
    ax.set_xlim(0, 105)
    ax.set_ylim(bottom=0)
    plt.tight_layout()
    plt.show()
    
    # Generate calibration points for vehicle_profiles.json
    cal_points = {}
    for pct in [20, 35, 50, 65, 80, 100]:
        cal_points[str(pct)] = round(speed_model(pct, k_fit), 1)
    print('\nCalibration points for vehicle_profiles.json:')
    print(json.dumps(cal_points, indent=2))
else:
    print('⚠️  Skipping speed model fit — not enough data.')
    k_fit = None
    cal_points = None

---
## 4. Model 2: Braking / Deceleration

**Goal:** Measure actual deceleration when motor goes from speed → 0.

**Current value:** `deceleration_cmss: 200.0` (guess). This affects every safety threshold.

**Method:** Find transitions where motor drops to 0, then measure how quickly the car actually stops using IMU + ToF.

In [ ]:
# Find braking events: motor was > 0, then becomes 0
df['motor_prev'] = df['motor'].shift(1)
brake_starts = df[(df['motor'] == 0) & (df['motor_prev'] > 10)].index

print(f'Found {len(brake_starts)} braking events (motor > 10% → 0)')

braking_events = []

for idx in brake_starts:
    # Get the speed just before braking
    pre_speed = df.loc[idx, 'motor_prev']
    
    # Look at the next ~20 samples (2 seconds at 10Hz) for deceleration
    window = df.loc[idx:idx+20].copy()
    window = window.dropna(subset=['tof_l', 'tof_r'])
    
    if len(window) < 5:
        continue
    
    window['front_dist'] = window[['tof_l', 'tof_r']].min(axis=1)
    window['velocity'] = -window['front_dist'].diff() / window['dt']
    window = window.dropna(subset=['velocity'])
    window = window[window['velocity'].between(-5, 80)]
    
    if len(window) >= 3:
        # Estimate initial velocity and deceleration
        initial_v = window['velocity'].iloc[0]
        
        # Find when velocity reaches ~0 (within noise)
        stopped = window[window['velocity'] < 1.0]
        if len(stopped) > 0:
            stop_time = stopped['t'].iloc[0] - window['t'].iloc[0]
            if stop_time > 0:
                decel = initial_v / stop_time  # cm/s²
                braking_events.append({
                    'pre_motor_pct': pre_speed,
                    'initial_velocity': initial_v,
                    'stop_time_s': stop_time,
                    'deceleration_cmss': decel,
                })

if braking_events:
    brake_df = pd.DataFrame(braking_events)
    print('\nBraking events:')
    print(brake_df.round(2))
    
    mean_decel = brake_df['deceleration_cmss'].mean()
    std_decel = brake_df['deceleration_cmss'].std()
    print(f'\nMeasured deceleration: {mean_decel:.1f} ± {std_decel:.1f} cm/s²')
    print(f'Current profile value: 200.0 cm/s²')
    print(f'Recommendation: {mean_decel:.0f} cm/s²')
    
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(range(len(brake_df)), brake_df['deceleration_cmss'], color='coral', alpha=0.8)
    ax.axhline(y=mean_decel, color='red', linewidth=2, linestyle='--', label=f'Mean: {mean_decel:.0f} cm/s²')
    ax.axhline(y=200, color='gray', linewidth=1, linestyle=':', label='Current profile: 200')
    ax.set_xlabel('Braking Event #')
    ax.set_ylabel('Deceleration (cm/s²)')
    ax.set_title('Measured Braking Deceleration')
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    mean_decel = None
    print('⚠️  No clean braking events found.')
    print('   Tip: Drive forward at steady speed, then stop suddenly. Repeat a few times.')

---
## 5. Model 3: Steering Response (Bicycle Model)

**Goal:** Map `(servo_angle, speed) → yaw_rate` for accurate path prediction.

**Current system:** Linear gain reduction (1.0 → 0.6) based on speed — a guess.

**Method:** Use gyroscope Z (yaw rate) correlated with servo angle and speed.

In [ ]:
# Filter for moving samples with valid IMU data
steer_df = df.copy()
steer_df = steer_df[steer_df['motor'] > 10]  # must be moving
steer_df = steer_df.dropna(subset=['gz'])
steer_df['steering_deflection'] = steer_df['servo'] - 90  # degrees from centre

print(f'Samples for steering model: {len(steer_df)}')

if len(steer_df) > 20:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Scatter: steering deflection vs yaw rate
    sc = axes[0].scatter(steer_df['steering_deflection'], steer_df['gz'],
                         c=steer_df['motor'], cmap='coolwarm', alpha=0.5, s=10)
    axes[0].set_xlabel('Steering Deflection (° from centre)')
    axes[0].set_ylabel('Yaw Rate (°/s)')
    axes[0].set_title('Steering Deflection vs Yaw Rate')
    axes[0].axhline(y=0, color='k', linewidth=0.5)
    axes[0].axvline(x=0, color='k', linewidth=0.5)
    plt.colorbar(sc, ax=axes[0], label='Motor %')
    
    # Group by steering bins
    steer_df['steer_bin'] = pd.cut(steer_df['steering_deflection'], 
                                    bins=[-60, -30, -15, -5, 5, 15, 30, 60],
                                    labels=['Hard L', 'Slight L', 'Near L', 'Centre', 
                                            'Near R', 'Slight R', 'Hard R'])
    steer_summary = steer_df.groupby('steer_bin', observed=True)['gz'].agg(['mean', 'std', 'count'])
    
    steer_summary['mean'].plot(kind='bar', ax=axes[1], color='steelblue', alpha=0.8)
    axes[1].set_xlabel('Steering Region')
    axes[1].set_ylabel('Mean Yaw Rate (°/s)')
    axes[1].set_title('Yaw Rate by Steering Region')
    axes[1].axhline(y=0, color='k', linewidth=0.5)
    plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=45)
    
    plt.tight_layout()
    plt.show()
    
    print('\nSteering response summary:')
    print(steer_summary.round(2))
else:
    print('⚠️  Not enough moving+steering samples.')
    print('   Tip: Drive with various steering angles at different speeds.')

In [ ]:
# Fit bicycle model: yaw_rate = (v / L) * tan(δ)
# where v = speed, L = wheelbase, δ = steering angle
# We fit L (effective wheelbase) as the free parameter

def bicycle_model(X, L_eff):
    """Bicycle model: yaw_rate = (speed / L) * tan(steering_rad)"""
    steering_deg, motor_pct = X
    steering_rad = np.radians(steering_deg)
    # Use our speed model if fitted, else use a simple linear estimate
    if k_fit is not None:
        speed = speed_model(motor_pct, k_fit)
    else:
        speed = motor_pct * 0.11  # fallback: profile default
    return np.degrees(speed / L_eff * np.tan(steering_rad))

if len(steer_df) > 20:
    # Filter for clean data (no extreme values)
    fit_df = steer_df[
        (steer_df['steering_deflection'].abs() > 3) &  # not dead-centre
        (steer_df['gz'].abs() < 200)  # reasonable yaw rates
    ].copy()
    
    if len(fit_df) > 10:
        X_data = (fit_df['steering_deflection'].values, fit_df['motor'].values)
        y_data = fit_df['gz'].values
        
        try:
            popt, pcov = curve_fit(bicycle_model, X_data, y_data, p0=[22.0], bounds=(5, 100))
            L_eff = popt[0]
            print(f'Fitted effective wheelbase: {L_eff:.1f} cm')
            print(f'Actual wheelbase (profile): 22.0 cm')
            print(f'\nThis means the car\'s steering is {"tighter" if L_eff < 22 else "wider"} than the physical wheelbase suggests.')
            
            # Predicted vs actual
            fit_df['predicted_gz'] = bicycle_model(X_data, L_eff)
            
            fig, ax = plt.subplots(figsize=(8, 6))
            ax.scatter(fit_df['gz'], fit_df['predicted_gz'], alpha=0.3, s=10)
            lims = [min(fit_df['gz'].min(), fit_df['predicted_gz'].min()),
                    max(fit_df['gz'].max(), fit_df['predicted_gz'].max())]
            ax.plot(lims, lims, 'r--', label='Perfect fit')
            ax.set_xlabel('Actual Yaw Rate (°/s)')
            ax.set_ylabel('Predicted Yaw Rate (°/s)')
            ax.set_title(f'Bicycle Model: L_eff = {L_eff:.1f} cm')
            ax.legend()
            plt.tight_layout()
            plt.show()
        except Exception as e:
            print(f'Bicycle model fit failed: {e}')
            L_eff = None
    else:
        print('⚠️  Not enough steering samples away from centre.')
        L_eff = None
else:
    L_eff = None

---
## 6. Model 4: Terrain Compensation

**Goal:** Measure how much speed is lost on inclines, compare to the current `sin(θ)` model.

**Current system:** `boost = sin(pitch) × MAX_INCLINE_BOOST` (linear in sin).

**Method:** Correlate pitch angle with actual speed at each motor %.

In [ ]:
terrain_df = df.copy()
terrain_df = terrain_df[terrain_df['motor'] > 10]
terrain_df = terrain_df.dropna(subset=['pitch', 'tof_l', 'tof_r'])
terrain_df['front_dist'] = terrain_df[['tof_l', 'tof_r']].min(axis=1)
terrain_df['velocity'] = -terrain_df['front_dist'].diff() / terrain_df['dt']
terrain_df = terrain_df.dropna(subset=['velocity'])
terrain_df = terrain_df[terrain_df['velocity'].between(0, 80)]

print(f'Samples for terrain model: {len(terrain_df)}')

if len(terrain_df) > 20:
    # Check if there's any meaningful pitch variation
    pitch_range = terrain_df['pitch'].max() - terrain_df['pitch'].min()
    print(f'Pitch range: {terrain_df["pitch"].min():.1f}° to {terrain_df["pitch"].max():.1f}° (range={pitch_range:.1f}°)')
    
    if pitch_range > 3:  # at least 3 degrees of variation
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        
        # Scatter: pitch vs velocity
        sc = axes[0].scatter(terrain_df['pitch'], terrain_df['velocity'],
                            c=terrain_df['motor'], cmap='viridis', alpha=0.4, s=10)
        axes[0].set_xlabel('Pitch (°) [+ = uphill]')
        axes[0].set_ylabel('Speed (cm/s)')
        axes[0].set_title('Speed vs Terrain Incline')
        plt.colorbar(sc, ax=axes[0], label='Motor %')
        
        # Bin by pitch and compute speed loss
        terrain_df['pitch_bin'] = pd.cut(terrain_df['pitch'], bins=8)
        terrain_summary = terrain_df.groupby('pitch_bin', observed=True).agg(
            mean_velocity=('velocity', 'mean'),
            mean_motor=('motor', 'mean'),
            count=('velocity', 'count')
        )
        terrain_summary = terrain_summary[terrain_summary['count'] >= 3]
        
        if len(terrain_summary) > 1:
            terrain_summary['mean_velocity'].plot(kind='bar', ax=axes[1], 
                                                   color='forestgreen', alpha=0.8)
            axes[1].set_xlabel('Pitch Range (°)')
            axes[1].set_ylabel('Mean Speed (cm/s)')
            axes[1].set_title('Speed by Terrain Angle')
            plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=45)
        
        plt.tight_layout()
        plt.show()
        
        print('\nTerrain impact summary:')
        print(terrain_summary.round(2))
    else:
        print('⚠️  Session was on mostly flat terrain — not enough incline variation.')
        print('   Tip: Drive on ramps or inclined surfaces.')
else:
    print('⚠️  Not enough data for terrain analysis.')

---
## 7. Export Updated Vehicle Profile

Generate an updated `vehicle_profiles.json` with the calibrated values.

In [ ]:
# Load current profile
PROFILES_PATH = Path('vehicle_profiles.json')
with open(PROFILES_PATH) as f:
    profiles = json.load(f)

# Which profile to update?
PROFILE_KEY = 'dual_motor'  # change to 'single_motor' if needed
profile = profiles[PROFILE_KEY]

print(f'Updating profile: {PROFILE_KEY}')
print(f'Description: {profile["description"]}')
print()

updates = {}

# Speed calibration
if cal_points:
    print('Speed calibration:')
    print(f'  Before: {profile["speed_calibration"]}')
    print(f'  After:  {cal_points}')
    updates['speed_calibration'] = cal_points

# Deceleration
if mean_decel is not None:
    old = profile['physics']['deceleration_cmss']
    new = round(mean_decel, 1)
    print(f'\nDeceleration:')
    print(f'  Before: {old} cm/s²')
    print(f'  After:  {new} cm/s²')
    updates['deceleration_cmss'] = new

if not updates:
    print('No models produced enough data for profile updates.')
    print('Collect more driving sessions and re-run.')
else:
    print(f'\n--- {len(updates)} update(s) ready ---')

In [ ]:
# Apply updates and save
SAVE = False  # <-- Set to True to actually write the file

if updates and SAVE:
    if 'speed_calibration' in updates:
        profile['speed_calibration'] = updates['speed_calibration']
    if 'deceleration_cmss' in updates:
        profile['physics']['deceleration_cmss'] = updates['deceleration_cmss']
    
    profiles[PROFILE_KEY] = profile
    
    # Write with backup
    backup_path = PROFILES_PATH.with_suffix('.json.bak')
    import shutil
    shutil.copy2(PROFILES_PATH, backup_path)
    print(f'Backup saved to {backup_path}')
    
    with open(PROFILES_PATH, 'w') as f:
        json.dump(profiles, f, indent=2)
    print(f'✓ Updated {PROFILES_PATH}')
    print('  Upload to Pico to use the new calibration.')
elif updates:
    print('Updates ready but SAVE=False. Set SAVE=True in the cell above to write.')
else:
    print('Nothing to save.')

---
## 8. Session Summary

In [ ]:
print('=' * 60)
print('SESSION MODELING SUMMARY')
print('=' * 60)
print(f'Log file:       {LOG_FILE}')
print(f'Duration:       {log_data["duration_s"]}s ({log_data["sample_count"]} samples @ {log_data["interval_ms"]}ms)')
print()

if k_fit is not None:
    print(f'✅ Speed Calibration:   k = {k_fit:.2f} cm/s at 100%')
else:
    print(f'⬜ Speed Calibration:   Not enough data')

if mean_decel is not None:
    print(f'✅ Braking Model:       decel = {mean_decel:.0f} cm/s² (was 200)')
else:
    print(f'⬜ Braking Model:       Not enough data')

if 'L_eff' in dir() and L_eff is not None:
    print(f'✅ Steering Model:      L_eff = {L_eff:.1f} cm (wheelbase=22.0)')
else:
    print(f'⬜ Steering Model:      Not enough data')

print(f'⬜ Terrain Model:       Requires slope driving data')

print()
print('TIPS FOR BETTER DATA:')
print('  1. Speed cal  → Drive straight toward a wall at 20%, 50%, 80%, 100%')
print('  2. Braking    → Drive forward, then stop suddenly (repeat 5+ times)')
print('  3. Steering   → Drive forward with different servo angles')
print('  4. Terrain    → Drive up/down ramps or inclined surfaces')
print('=' * 60)